In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\sreej\OneDrive\Desktop\Power BI PROJECT\RAW_Global_Nike.csv")

print(df.head())

  snapshot_date country_code     product_name model_number currency  \
0    2026-03-19           US  "Grateful Duck"  NIKGD001K01      USD   
1    2026-03-19           US  "Grateful Duck"  NIKGD001K01      USD   
2    2026-03-19           US  "Grateful Duck"  NIKGD001K01      USD   
3    2026-03-19           US  "Grateful Duck"  NIKGD001K01      USD   
4    2026-03-19           US  "Grateful Duck"  NIKGD001K01      USD   

   price_local  sale_price_local gender_segment size_label category  ...  \
0        110.0               NaN            MEN          L  APPAREL  ...   
1        110.0               NaN            MEN          M  APPAREL  ...   
2        110.0               NaN            MEN          S  APPAREL  ...   
3        110.0               NaN            MEN         XL  APPAREL  ...   
4        110.0               NaN            MEN        XXL  APPAREL  ...   

                                       canonical_url  \
0  https://www.nike.com/t/grateful-duck-mens-tie-...   
1  h

In [2]:
print(df.info())
print(df.describe())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1447795 entries, 0 to 1447794
Data columns (total 35 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   snapshot_date          1447795 non-null  object 
 1   country_code           1447795 non-null  object 
 2   product_name           1447795 non-null  object 
 3   model_number           1447795 non-null  object 
 4   currency               1447795 non-null  object 
 5   price_local            1447795 non-null  float64
 6   sale_price_local       210687 non-null   float64
 7   gender_segment         1441048 non-null  object 
 8   size_label             1447793 non-null  object 
 9   category               1447795 non-null  object 
 10  subcategory            1447767 non-null  object 
 11  product_id             1447795 non-null  object 
 12  sku                    1447793 non-null  object 
 13  style_color            1447795 non-null  object 
 14  brand_name        

In [3]:
# Convert snapshot_date to datetime
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'])

# Drop completely empty columns
df = df.drop(columns=['size_count', 'available_size_count', 'record_source'])

# Optional: drop sparse columns if not needed
df = df.drop(columns=['sport_tags'])

# Fill missing gender_segment with 'Unknown'
df['gender_segment'] = df['gender_segment'].fillna('Unknown')

# Fill missing brand_name with 'Unknown'
df['brand_name'] = df['brand_name'].fillna('Unknown')

# Fill missing sale_price_local with price_local (assumes not on sale)
df['sale_price_local'] = df['sale_price_local'].fillna(df['price_local'])

In [4]:
print(df['country_code'].nunique())
print(df['category'].nunique())

45
5


In [5]:
print(df.duplicated(subset=['snapshot_date','product_id']).sum())

1418077


In [6]:
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'])

In [7]:
print(df['country_code'].nunique())
print(df['category'].unique())
print(df['gender_segment'].unique())

45
['APPAREL' 'EQUIPMENT' 'FOOTWEAR' 'DIGITAL_GIFT_CARD' 'PHYSICAL_GIFT_CARD']
['MEN' 'WOMEN' 'MEN|WOMEN' 'BOYS|GIRLS' 'BOYS' 'Unknown'
 'MEN|BOYS|WOMEN|GIRLS' 'GIRLS' 'MEN|WOMEN|GIRLS' 'WOMEN|GIRLS'
 'MEN|BOYS|GIRLS' 'BOYS|WOMEN|GIRLS' 'GIRLS|BOYS' 'WOMEN|MEN']


In [8]:
df['discount'] = df['price_local'] - df['sale_price_local']

In [9]:
print(df[['price_local', 'sale_price_local', 'discount']].head())

   price_local  sale_price_local  discount
0        110.0             110.0       0.0
1        110.0             110.0       0.0
2        110.0             110.0       0.0
3        110.0             110.0       0.0
4        110.0             110.0       0.0


In [10]:
df['discount'] = df['price_local'] - df['sale_price_local']

In [11]:
df['sale_price_local'] = df['sale_price_local'].fillna(df['price_local'])

In [12]:
df['discount'] = df['price_local'] - df['sale_price_local']

In [13]:
print(df[['price_local', 'sale_price_local', 'discount']].head())

   price_local  sale_price_local  discount
0        110.0             110.0       0.0
1        110.0             110.0       0.0
2        110.0             110.0       0.0
3        110.0             110.0       0.0
4        110.0             110.0       0.0


In [14]:
df['is_on_sale'] = df['price_local'] != df['sale_price_local']

In [15]:
print(df['is_on_sale'].value_counts())

is_on_sale
False    1237108
True      210687
Name: count, dtype: int64


In [16]:
df.groupby('category')['is_on_sale'].mean()

category
APPAREL               0.130009
DIGITAL_GIFT_CARD     0.000000
EQUIPMENT             0.090379
FOOTWEAR              0.162444
PHYSICAL_GIFT_CARD    0.000000
Name: is_on_sale, dtype: float64

In [17]:
df['discount_pct'] = (df['price_local'] - df['sale_price_local']) / df['price_local'] * 100

In [18]:
df.groupby('category')['discount_pct'].describe()

,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
APPAREL,696577.0,-4.447152,12.330068,-103.315608,0.0,0.0,0.0,0.0
DIGITAL_GIFT_CARD,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EQUIPMENT,26389.0,-2.729593,9.611276,-100.000000,0.0,0.0,0.0,0.0
FOOTWEAR,724811.0,-5.560476,13.847482,-100.160256,0.0,0.0,0.0,0.0
PHYSICAL_GIFT_CARD,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
# Create Products table
products = df[['product_id','product_name','model_number','category','subcategory','brand_name','gender_segment']].drop_duplicates()

# Create Prices table
prices = df[['product_id','snapshot_date','currency','price_local','sale_price_local','discount','discount_pct','is_on_sale']]

# Create Inventory table
inventory = df[['product_id','available','size_label','sku','availability_level']]

In [26]:
products.to_csv("C:/Users/sreej/OneDrive/Desktop/Power BI PROJECT/NIKE SALES PROJECT/products.csv", index=False)

prices.to_csv("C:/Users/sreej/OneDrive/Desktop/Power BI PROJECT/NIKE SALES PROJECT/prices.csv", index=False)

inventory.to_csv("C:/Users/sreej/OneDrive/Desktop/Power BI PROJECT/NIKE SALES PROJECT/inventory.csv", index=False)

In [27]:
# Discount percentage (if not already)
df['discount_pct'] = (df['price_local'] - df['sale_price_local']) / df['price_local'] * 100

# Price bucket (very important for storytelling)
df['price_bucket'] = pd.cut(df['price_local'],
                           bins=[0,50,100,200,500,1000],
                           labels=['Low','Budget','Mid','Premium','Luxury'])

In [30]:
df[df['discount'] < 0][['price_local', 'sale_price_local', 'discount']].head()

,price_local,sale_price_local,discount
50,21.97,30.0,-8.03
51,21.97,30.0,-8.03
52,21.97,30.0,-8.03
53,21.97,30.0,-8.03
54,21.97,30.0,-8.03


In [31]:
(df['discount'] < 0).sum()

210687

In [32]:
# Check how many
print((df['discount'] < 0).sum())

# Remove them
df = df[df['discount'] >= 0]

210687


In [33]:
df['discount_pct'] = (df['price_local'] - df['sale_price_local']) / df['price_local'] * 100

In [34]:
df[df['discount'] < 0][['price_local','sale_price_local','country_code','category']].head(20)

,price_local,sale_price_local,country_code,category


In [35]:
df['discount'].describe()

count    1237108.0
mean           0.0
std            0.0
min            0.0
25%            0.0
50%            0.0
75%            0.0
max            0.0
Name: discount, dtype: float64

In [36]:
df['sale_price_local'] = df[['price_local','sale_price_local']].min(axis=1)
df['discount'] = df['price_local'] - df['sale_price_local']

In [37]:
df['discount_pct'] = (df['price_local'] - df['sale_price_local']) / df['price_local'] * 100
(df['discount'] < 0).sum()

0

In [38]:
df['sale_price_local'] = df[['price_local','sale_price_local']].min(axis=1)

In [39]:
df[df['sale_price_local'].notna()][['price_local','sale_price_local','category']].head(20)

,price_local,sale_price_local,category
0,110.0,110.0,APPAREL
1,110.0,110.0,APPAREL
2,110.0,110.0,APPAREL
3,110.0,110.0,APPAREL
4,110.0,110.0,APPAREL
5,130.0,130.0,APPAREL
6,130.0,130.0,APPAREL
7,130.0,130.0,APPAREL
8,130.0,130.0,APPAREL
9,130.0,130.0,APPAREL


In [40]:
# Convert to datetime, coerce errors to NaT (null)
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'], errors='coerce')

# Remove rows where snapshot_date could not be parsed
df_clean = df.dropna(subset=['snapshot_date'])

# Export final CSV
df_clean.to_csv("C:/Users/sreej/OneDrive/Desktop/Power BI PROJECT/final_nike_data_clean.csv", index=False)

In [29]:
df.to_csv("C:/Users/sreej/OneDrive/Desktop/Power BI PROJECT/NIKE SALES PROJECT/final_nike_data.csv", index=False)